In [ ]:
%%bash
# We pin protobuf<6 to avoid conflict warnings with Colab's pre-installed Google libraries
pip install -q "protobuf<6.0.0dev"
pip install -q fast-plate-ocr[train,onnx-gpu]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r "/content/drive/MyDrive/train_data" /content/train_data


In [ ]:
!ls -lh /content/train_data
!ls -1 /content/train_data/images | wc -l


total 1.1M
drwx------ 2 root root 228K Sep 23 18:09 crops_clahe
drwx------ 2 root root 248K Sep 23 18:12 images
-rw------- 1 root root 191K Sep 23 18:06 rec_gt_train.txt
-rw------- 1 root root  34K Sep 23 18:06 rec_gt_val.txt
-rw------- 1 root root 163K Sep 23 18:06 train.csv
-rw------- 1 root root 163K Sep 23 18:06 train_list.txt
-rw------- 1 root root  29K Sep 23 18:06 val.csv
-rw------- 1 root root  29K Sep 23 18:06 val_list.txt
6745


In [ ]:
import os
import csv
import shutil

DRIVE_DATA_DIR = "/content/drive/MyDrive/train_data"
LOCAL_DATA_DIR = "/content/train_data"

if not os.path.exists(DRIVE_DATA_DIR):
    raise FileNotFoundError(f"⚠️ Could not find {DRIVE_DATA_DIR}. Ensure Google Drive is mounted correctly.")

# Copy dataset to local Colab SSD for maximum I/O performance
if not os.path.exists(LOCAL_DATA_DIR):
    print("Copying dataset to local Colab SSD for high-speed training...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Dataset copied successfully!")

DATA_DIR = LOCAL_DATA_DIR if os.path.exists(LOCAL_DATA_DIR) else DRIVE_DATA_DIR

def convert_tsv_to_csv(tsv_rel_path, csv_out_path):
    tsv_path = os.path.join(DATA_DIR, tsv_rel_path)
    count = 0
    with open(tsv_path, 'r', encoding='utf-8') as fin, \
         open(csv_out_path, 'w', newline='', encoding='utf-8') as fout:
        writer = csv.writer(fout)
        writer.writerow(['image_path', 'plate_text'])
        for line in fin:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                img_path, label = parts
                # Point to raw images (benchmark proved raw beats CLAHE)
                img_path = img_path.replace('crops_clahe/', 'images/')
                writer.writerow([img_path, label])
                count += 1
    print(f"Generated {csv_out_path} ({count} samples)")

# CSVs are stored inside DATA_DIR so image paths ('images/...') resolve cleanly
convert_tsv_to_csv('rec_gt_train.txt', os.path.join(DATA_DIR, 'train_annotations.csv'))
convert_tsv_to_csv('rec_gt_val.txt', os.path.join(DATA_DIR, 'val_annotations.csv'))
print("==> Annotations generated OK!")

Generated /content/train_data/train_annotations.csv (5733 samples)
Generated /content/train_data/val_annotations.csv (1012 samples)
==> Annotations generated OK!


In [ ]:
%%bash
wget -q -O /content/model_config.yaml \
  https://github.com/ankandrew/cnn-ocr-lp/releases/download/arg-plates/cct_s_v2_global_model_config.yaml

wget -q -O /content/plate_config.yaml \
  https://github.com/ankandrew/cnn-ocr-lp/releases/download/arg-plates/cct_s_v2_global_plate_config.yaml

wget -q -O /content/cct_s_v2_global.keras \
  https://github.com/ankandrew/cnn-ocr-lp/releases/download/arg-plates/cct_s_v2_global.keras

echo "==> Downloaded model_config.yaml, plate_config.yaml, and cct_s_v2_global.keras!"

==> Downloaded model_config.yaml, plate_config.yaml, and cct_s_v2_global.keras!


In [ ]:
import os
import csv

# We assume you have mounted your Google Drive first!
# from google.colab import drive
# drive.mount('/content/drive')

def tsv_to_csv(tsv_path, csv_path):
    if not os.path.exists(tsv_path):
        raise FileNotFoundError(f"⚠️ ERROR: {tsv_path} not found! Check your Google Drive mount.")

    with open(tsv_path, 'r', encoding='utf-8') as fin, \
         open(csv_path, 'w', newline='', encoding='utf-8') as fout:
        writer = csv.writer(fout)
        writer.writerow(['image_path', 'plate_text'])
        for line in fin:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                img_path, label = parts

                # The TSV uses 'crops_clahe/img_...'. We want to use the RAW images in 'images/'
                img_path = img_path.replace('crops_clahe/', 'images/')

                # Point directly to your mounted Google Drive
                writer.writerow([f"/content/drive/MyDrive/train_data/{img_path}", label])

# Read from Google Drive, output CSVs to the local /content/ space
tsv_to_csv('/content/drive/MyDrive/train_data/rec_gt_train.txt', '/content/train_annotations.csv')
tsv_to_csv('/content/drive/MyDrive/train_data/rec_gt_val.txt', '/content/val_annotations.csv')
print("==> Generated train_annotations.csv and val_annotations.csv successfully!")

==> Generated train_annotations.csv and val_annotations.csv successfully!


In [ ]:
from fast_plate_ocr.train.model.config import load_plate_config_from_yaml
from fast_plate_ocr.train.model.model_schema import load_model_config_from_yaml

plate_cfg = load_plate_config_from_yaml('/content/plate_config.yaml')
model_cfg = load_model_config_from_yaml('/content/model_config.yaml')

print(f"✅ PlateConfig Validated: alphabet_size={plate_cfg.vocabulary_size}, pad_char={repr(plate_cfg.pad_char)}, slots={plate_cfg.max_plate_slots}, shape=({plate_cfg.img_height}, {plate_cfg.img_width})")
print(f"✅ ModelConfig Validated: model={model_cfg.model}")

✅ PlateConfig Validated: alphabet_size=37, pad_char='_', slots=10, shape=(64, 128)
✅ ModelConfig Validated: model=cct


In [ ]:
%%bash
export KERAS_BACKEND=torch

ANNOTATIONS="/content/train_data/train_annotations.csv"
VAL_ANNOTATIONS="/content/train_data/val_annotations.csv"

# Fallback to Google Drive if local copy was skipped
if [ ! -f "$ANNOTATIONS" ]; then
  ANNOTATIONS="/content/drive/MyDrive/train_data/train_annotations.csv"
  VAL_ANNOTATIONS="/content/drive/MyDrive/train_data/val_annotations.csv"
fi

echo "Using annotations: $ANNOTATIONS"
echo "Using validation:  $VAL_ANNOTATIONS"

fast-plate-ocr train \
  --model-config-file /content/model_config.yaml \
  --plate-config-file /content/plate_config.yaml \
  --annotations "$ANNOTATIONS" \
  --val-annotations "$VAL_ANNOTATIONS" \
  --weights-path /content/cct_s_v2_global.keras \
  --epochs 80 \
  --batch-size 64 \
  --lr 0.0001 \
  --warmup-fraction 0.05 \
  --final-lr-factor 0.01 \
  --weight-decay 0.01 \
  --output-dir /content/output

Process is terminated.


In [ ]:
import os
import sys
import subprocess

# 1. Force PyTorch backend and disable stdout buffering
os.environ["KERAS_BACKEND"] = "torch"
os.environ["PYTHONUNBUFFERED"] = "1"

# 2. Check dataset path (prefer fast NVMe SSD)
annotations = "/content/train_data/train_annotations.csv"
val_annotations = "/content/train_data/val_annotations.csv"

if not os.path.exists(annotations):
    print("⚠️ Local /content/train_data not found, falling back to Google Drive...")
    annotations = "/content/drive/MyDrive/train_data/train_annotations.csv"
    val_annotations = "/content/drive/MyDrive/train_data/val_annotations.csv"
    print("💡 TIP: Copying to /content/train_data first makes training 15x faster!")
else:
    print("✅ Using high-speed local dataset at /content/train_data")

cmd = [
    "fast-plate-ocr", "train",
    "--model-config-file", "/content/model_config.yaml",
    "--plate-config-file", "/content/plate_config.yaml",
    "--annotations", annotations,
    "--val-annotations", val_annotations,
    "--weights-path", "/content/cct_s_v2_global.keras",
    "--epochs", "80",
    "--batch-size", "64",
    "--lr", "0.0001",
    "--warmup-fraction", "0.05",
    "--final-lr-factor", "0.01",
    "--weight-decay", "0.01",
    "--output-dir", "/content/output"
]

print("🚀 Starting training with live terminal output:\n" + "=" * 60)

# pty connects directly as an interactive terminal so progress bars render live
try:
    import pty
    master, slave = pty.openpty()
    proc = subprocess.Popen(
        cmd,
        stdin=slave,
        stdout=slave,
        stderr=slave,
        close_fds=True
    )
    os.close(slave)

    while True:
        try:
            data = os.read(master, 1024)
            if not data:
                break
            sys.stdout.write(data.decode("utf-8", errors="replace"))
            sys.stdout.flush()
        except OSError:
            break

    os.close(master)
    proc.wait()
except (ImportError, AttributeError):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()

if proc.returncode == 0:
    print("\n" + "=" * 60 + "\n🎉 Training finished successfully!")
else:
    print(f"\n⚠️ Process exited with code {proc.returncode}")


✅ Using high-speed local dataset at /content/train_data
🚀 Starting training with live terminal output:


                                 CLI Training Parameters                        
╭─────────────────────────┬─────────────────────────────────────────────────────
│ Parameter               │ Details                                             
├─────────────────────────┼─────────────────────────────────────────────────────
│ model_config_file       │ '/content/model_config.yaml'                        
│ plate_config_file       │ '/content/plate_config.yaml'                        
│ annotations             │ '/content/train_data/train_annotations.csv'         
│ val_annotations         │ '/content/train_data/val_annotations.csv'           
│ validate_dataset        │ 'off'                                               
│ validation_freq         │ 1                                                   
│ augmentation_path       │ None                                                
│ lr

In [ ]:
%%bash
# 1. Install missing exporter dependency
pip install -q onnxscript

# 2. Locate the best checkpoint produced by training
BEST_KERAS=$(find /content/output -name "best.keras" | sort | tail -n 1)

echo "Exporting: $BEST_KERAS to ONNX..."

# 3. Export to ONNX
fast-plate-ocr export \
  --model "$BEST_KERAS" \
  --plate-config-file /content/plate_config.yaml \
  --format onnx \
  --save-dir /content

# 4. Verify file was generated
if [ -f "/content/best.onnx" ]; then
  echo "==> Export complete! Your model is saved at /content/best.onnx"
  ls -lh /content/best.onnx
else
  echo "⚠️ Export failed: /content/best.onnx was not created."
fi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 16.2 MB/s eta 0:00:00
Exporting: /content/output/2026-09-23_18-18-00/best.keras to ONNX...
⚠️ Export failed: /content/best.onnx was not created.


/usr/local/lib/python3.13/dist-packages/torch/_export/non_strict_utils.py:1159: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return func(*args, **kwargs)
W0923 19:48:21.284000 38208 torch/fx/experimental/symbolic_shapes.py:8255] Unable to find user code corresponding to {u0}
/usr/local/lib/python3.13/dist-packages/torch/_export/non_strict_utils.py:1159: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Trigger

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import glob, pathlib, inspect, keras
import fast_plate_ocr.train.model.layers as layers
from fast_plate_ocr.train.model.config import load_plate_config_from_yaml
from fast_plate_ocr.train.utilities.utils import load_keras_model
from fast_plate_ocr.cli.export import export_onnx

# Check what the layer currently does before patching
print(inspect.getsource(layers.PositionEmbedding.call))

def _static_call(self, inputs, start_index=0):
    seq_len = inputs.shape[1]  # static int, only the batch dim is dynamic
    pos = self.position_embeddings[start_index:start_index + seq_len]
    return keras.ops.broadcast_to(pos, keras.ops.shape(inputs))

layers.PositionEmbedding.call = _static_call

model_path = pathlib.Path(sorted(glob.glob("/content/output/**/best.keras", recursive=True))[-1])
plate_config = load_plate_config_from_yaml("/content/plate_config.yaml")
model = load_keras_model(model_path, plate_config)

export_onnx(
    model=model,
    plate_config=plate_config,
    out_file=pathlib.Path("/content/best.onnx"),
    simplify=True,
    dynamic_batch=True,
    skip_validation=False,
    onnx_input_dtype="uint8",
    onnx_data_format="channels_last",
)

    def call(self, inputs, start_index=0):
        shape = keras.ops.shape(inputs)
        feature_length = shape[-1]
        sequence_length = shape[-2]
        # trim to match the length of the input sequence, which might be less than the
        # sequence_length of the layer.
        position_embeddings = keras.ops.convert_to_tensor(self.position_embeddings)
        position_embeddings = keras.ops.slice(
            position_embeddings,
            (start_index, 0),
            (sequence_length, feature_length),
        )
        return keras.ops.broadcast_to(position_embeddings, shape)



/usr/local/lib/python3.13/dist-packages/keras/src/export/tf2onnx_lib.py:20: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:
import keras
import fast_plate_ocr.train.model.layers as layers

def _static_call(self, inputs, start_index=0):
    shape = keras.ops.shape(inputs)
    seq_len = inputs.shape[-2]  # static Python int; only the batch dim is dynamic
    pos = keras.ops.convert_to_tensor(self.position_embeddings)
    pos = pos[start_index:start_index + seq_len]  # same trim as the original
    return keras.ops.broadcast_to(pos, shape)

layers.PositionEmbedding.call = _static_call

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import glob, pathlib
from fast_plate_ocr.train.model.config import load_plate_config_from_yaml
from fast_plate_ocr.train.utilities.utils import load_keras_model
from fast_plate_ocr.cli.export import export_onnx

# ... apply the patch above first ...

model_path = pathlib.Path(sorted(glob.glob("/content/output/**/best.keras", recursive=True))[-1])
plate_config = load_plate_config_from_yaml("/content/plate_config.yaml")
model = load_keras_model(model_path, plate_config)

export_onnx(
    model=model,
    plate_config=plate_config,
    out_file=pathlib.Path("/content/best.onnx"),
    simplify=True,
    dynamic_batch=True,
    skip_validation=False,   # keeps the built-in output comparison
    onnx_input_dtype="uint8",
    onnx_data_format="channels_last",
)

In [ ]:
import os, glob
print(os.path.exists("/content/best.onnx"))
print(glob.glob("/content/*.onnx"))
if os.path.exists("/content/best.onnx"):
    print(os.path.getsize("/content/best.onnx") / 1e6, "MB")

True
['/content/best.onnx']
4.901659 MB


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
import onnxruntime as ort

sess = ort.InferenceSession("/content/best.onnx")
for i in sess.get_inputs():
    print("input:", i.name, i.shape, i.type)
for o in sess.get_outputs():
    print("output:", o.name, o.shape, o.type)

input: input ['unk__373', 64, 128, 3] tensor(uint8)
output: plate ['unk__374', 10, 37] tensor(float)


In [ ]:
import inspect
import fast_plate_ocr.inference.plate_recognizer as pr

print([n for n, _ in inspect.getmembers(pr, inspect.isclass)])
print(inspect.getsource(pr))

['Console', 'LicensePlateRecognizer', 'Panel', 'PlateConfig', 'PlatePrediction', 'Sequence', 'Table', 'Text']
"""
ONNX Runtime inference for license plate recognition.
"""

import logging
import pathlib
from collections.abc import Sequence
from typing import Literal

import numpy as np
import numpy.typing as npt

try:
    import onnxruntime as ort
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "ONNX Runtime is not installed. Run: pip install 'fast-plate-ocr[onnx]' (or [onnx-gpu], etc.)"
    ) from e
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

from fast_plate_ocr.core.process import (
    postprocess_output,
    preprocess_image,
    read_and_resize_plate_image,
    resize_image,
)
from fast_plate_ocr.core.types import BatchArray, BatchOrImgLike, ImgLike, PathLike, PlatePrediction
from fast_plate_ocr.core.utils import measure_time
from fast_plate_ocr.inference import hub
from fast_plate_o

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import glob, inspect, pathlib
import pandas as pd

# 1. CSV layout
df = pd.read_csv("/content/train_data/val_annotations.csv")
print(df.columns.tolist(), len(df))
print(df.head(3).to_string())

# 2. Pre/post-processing source
import fast_plate_ocr.core.process as proc
print(inspect.getsource(proc.preprocess_image))
print(inspect.getsource(proc.postprocess_output))

# 3. What the export validation does
import fast_plate_ocr.cli.export as ex
for i, l in enumerate(open(ex.__file__).read().splitlines(), 1):
    if "validat" in l.lower():
        print(i, l)

# 4. Files training left behind (metrics/logs may be in here)
for f in sorted(glob.glob("/content/output/**/*", recursive=True)):
    print(f)

# 5. Keras model input spec
from fast_plate_ocr.train.model.config import load_plate_config_from_yaml
from fast_plate_ocr.train.utilities.utils import load_keras_model
mp = pathlib.Path(sorted(glob.glob("/content/output/**/best.keras", recursive=True))[-1])
m = load_keras_model(mp, load_plate_config_from_yaml("/content/plate_config.yaml"))
print(m.input_shape, m.input.dtype, [type(l).__name__ for l in m.layers[:4]])

['image_path', 'plate_text'] 1012
             image_path plate_text
0  images/img_06677.jpg    LES7148
1  images/img_05842.jpg    LES7626
2  images/img_06182.jpg     BPZ345
def preprocess_image(images: np.ndarray) -> np.ndarray:
    """
    Converts image data to the format expected by the model.

    The model itself handles pixel-value normalisation, so this function only ensures the
    batch-dimension and dtype are correct.

    Args:
        images: Image or batch of images with shape ``(H, W, C)`` or ``(N, H, W, C)``.

    Returns:
        A NumPy array with shape ``(N, H, W, C)`` and dtype ``uint8``.

    Raises:
        ValueError: If the input does not have 3 or 4 dimensions.
    """
    # single sample (H, W, C)
    if images.ndim == 3:
        images = np.expand_dims(images, axis=0)

    if images.ndim != 4:
        raise ValueError("Expected input of shape (N, H, W, C).")

    return images.astype(np.uint8)

def postprocess_output(
    model_output: np.ndarray,
    max_pla

In [ ]:
import numpy as np

expected = cfg.max_plate_slots * len(cfg.alphabet)

def get_plate(k, n):
    """Return the Keras output whose per-sample size matches the plate head."""
    if isinstance(k, dict):
        cands = list(k.values())
    elif isinstance(k, (list, tuple)):
        cands = list(k)
    else:
        cands = [k]
    cands = [np.asarray(c) for c in cands]
    for c in cands:
        if c.size == n * expected:
            return c
    raise ValueError(f"No output with per-sample size {expected}: "
                     f"{[(type(k).__name__, c.shape) for c in cands]}")

onnx_txt, keras_txt, max_diff = [], [], 0.0
for i in range(0, len(paths), 64):
    x = _load_image_from_source([str(p) for p in paths[i:i+64]], cfg)
    o = rec.model.run(["plate"], {"input": x})[0]
    k_raw = model.predict(x.astype("float32"), verbose=0)
    if i == 0:
        print("keras predict returned:", type(k_raw).__name__,
              {kk: np.asarray(v).shape for kk, v in k_raw.items()} if isinstance(k_raw, dict) else "")
    k = get_plate(k_raw, len(x))
    max_diff = max(max_diff, float(np.abs(o.reshape(len(x), -1) - k.reshape(len(x), -1)).max()))
    onnx_txt += decode(o)
    keras_txt += decode(k)

acc = lambda pred: np.mean([a == b for a, b in zip(pred, truth)])
print(f"ONNX  exact-match : {acc(onnx_txt):.4f}")
print(f"Keras exact-match : {acc(keras_txt):.4f}   (training log best: 0.8735)")
print(f"ONNX == Keras strings: {np.mean([a == b for a, b in zip(onnx_txt, keras_txt)]):.4f}")
print(f"max |ONNX - Keras| raw output diff: {max_diff:.2e}")
diffs = [(i, keras_txt[i], onnx_txt[i], truth[i]) for i in range(len(truth)) if keras_txt[i] != onnx_txt[i]]
print("disagreements:", len(diffs), "(index, keras, onnx, truth)")
print(diffs[:10])

keras predict returned: dict {'plate': (64, 10, 37)}


NameError: name 'truth' is not defined